In [ ]:
"""Connect to Kafka
→ read stock-trades-raw
→ convert Kafka key/value from bytes to strings
→ print records in the terminal"""

from pyspark.sql import SparkSession


KAFKA_BOOTSTRAP_SERVERS = "YOUR_KAFKA_PRIVATE_IP:9092"
KAFKA_TOPIC = "stock-trades-raw"

CHECKPOINT_LOCATION = (
    "/home/ubuntu/stock-market-streaming/"
    "checkpoints/raw_console_test"
)


def main() -> None:
    spark = (
        SparkSession.builder
        .appName("StockMarketRawKafkaConsumer")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    print(
        f"Connecting to Kafka: {KAFKA_BOOTSTRAP_SERVERS}"
    )
    print(
        f"Reading topic: {KAFKA_TOPIC}"
    )

    kafka_stream = (
        spark.readStream
        .format("kafka")
        .option(
            "kafka.bootstrap.servers",
            KAFKA_BOOTSTRAP_SERVERS,
        )
        .option("subscribe", KAFKA_TOPIC)
        .option("startingOffsets", "earliest")
        .load()
    )

    readable_stream = kafka_stream.selectExpr(
        "CAST(key AS STRING) AS message_key",
        "CAST(value AS STRING) AS message_value",
        "topic",
        "partition",
        "offset",
        "timestamp AS kafka_timestamp",
    )

    query = (
        readable_stream.writeStream
        .format("console")
        .outputMode("append")
        .option("truncate", "false")
        .option(
            "checkpointLocation",
            CHECKPOINT_LOCATION,
        )
        .trigger(processingTime="5 seconds")
        .start()
    )

    print("Spark Kafka consumer is running.")
    print("Press Ctrl+C to stop it.")

    query.awaitTermination()


if __name__ == "__main__":
    main()